In [ ]:
!python3 -m pip install pymupdf

In [ ]:
import bisect
import math

class P2Median:
    def __init__(self):
        # target quantile p = 0.5 for the median
        self.p = 0.5
        
        # buffers and markers
        self._buffer = []         # first 5 observations, to initialize
        self.n = [0]*5            # positions of markers
        self.q = [0.0]*5          # heights of markers (estimates of order statistics)
        self.np = [0.0]*5         # desired positions
        # increments for desired positions: [0, p/2, p, (1+p)/2, 1]
        self.dn = [0.0, self.p/2, self.p, (1+self.p)/2, 1.0]
        self.count = 0            # total number of points processed

    def insert(self, x):
        # Phase 0: buffer up first 5 points
        if self.count < 5:
            bisect.insort(self._buffer, x)
            self.count += 1
            if self.count == 5:
                # initialize markers
                for i in range(5):
                    self.q[i] = self._buffer[i]
                    self.n[i] = i + 1
                    # desired positions np:
                self.np = [
                    1,
                    1 + 2*self.p,
                    1 + 4*self.p,
                    3 + 2*self.p,
                    5
                ]
            return

        # Phase 1: update markers
        self.count += 1
        # find k: the cell in which x belongs
        k = 0
        if x < self.q[0]:
            self.q[0] = x
            k = 0
        elif x >= self.q[4]:
            self.q[4] = x
            k = 3
        else:
            for i in range(4):
                if self.q[i] <= x < self.q[i+1]:
                    k = i
                    break

        # increment positions of markers above k
        for i in range(k+1, 5):
            self.n[i] += 1
        # update desired positions
        for i in range(5):
            self.np[i] += self.dn[i]

        # adjust heights q[1..3]
        for i in range(1, 4):
            d = self.np[i] - self.n[i]
            if (d >= 1 and (self.n[i+1] - self.n[i]) > 1) or \
               (d <= -1 and (self.n[i] - self.n[i-1]) > 1):
                d_sign = int(math.copysign(1, d))
                # parabolic prediction
                num = (self.n[i] - self.n[i-1] + d_sign) * (self.q[i+1] - self.q[i]) / (self.n[i+1] - self.n[i]) \
                    + (self.n[i+1] - self.n[i] - d_sign) * (self.q[i] - self.q[i-1]) / (self.n[i] - self.n[i-1])
                qp = self.q[i] + d_sign * num / (self.n[i+1] - self.n[i-1])
                # accept parabolic if in bounds, else linear
                if self.q[i-1] < qp < self.q[i+1]:
                    self.q[i] = qp
                else:
                    # linear
                    self.q[i] += d_sign * (self.q[i + d_sign] - self.q[i]) / (self.n[i + d_sign] - self.n[i])
                self.n[i] += d_sign

    def median(self):
        if self.count < 5:
            # not enough data yet: return median of buffer
            return float(self._buffer[len(self._buffer)//2])
        return float(self.q[2])


stream = P2Median()
data = [5, 1, 9, 3, 14, 7, 6, 2, 10, 8, 11]
for x in data:
    stream.insert(x)
    print(f"Inserted {x:2d}, approx. median = {stream.median():.2f}")


In [ ]:
extracted_text = extract_text_from_pdf(fname)
print(extracted_text)

In [ ]:
import bm25s
import Stemmer  # optional: for stemming

# Create your corpus here
corpus = [
"AieRC _XAie_DmaSetLock(XAie_DmaDesc *DmaDesc, XAie_Lock Acq, XAie_Lock Rel,\
		u8 AcqEn, u8 RelEn);",
"AieRC _XAie_ShimDmaWriteBd(XAie_DevInst *DevInst , XAie_DmaDesc *DmaDesc,\
		XAie_LocType Loc, u8 BdNum);",
"AieRC _XAie_ShimDmaReadBd(XAie_DevInst *DevInst , XAie_DmaDesc *DmaDesc,\
		XAie_LocType Loc, u8 BdNum);",
"AieRC _XAie_TileDmaWriteBd(XAie_DevInst *DevInst , XAie_DmaDesc *DmaDesc,\
		XAie_LocType Loc, u8 BdNum);",
"AieRC _XAie_TileDmaReadBd(XAie_DevInst *DevInst , XAie_DmaDesc *DmaDesc,\
		XAie_LocType Loc, u8 BdNum);",
]

# optional: create a stemmer
stemmer = Stemmer.Stemmer("english")

# Tokenize the corpus and only keep the ids (faster and saves memory)
corpus_tokens = bm25s.tokenize(corpus, stopwords="en", stemmer=stemmer)

# Create the BM25 model and index the corpus
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Query the corpus
query = "shimdmawrite?"
query_tokens = bm25s.tokenize(query, stemmer=stemmer)

In [ ]:
query_tokens

In [ ]:
results, scores = retriever.retrieve(query_tokens, k=2)

for i in range(results.shape[1]):
    doc, score = results[0, i], scores[0, i]
    print(f"Rank {i+1} (score: {score:.2f}): {corpus[doc]}")

In [ ]:
from tree_sitter import Language, Parser
import tree_sitter_c as tsc
import re

# Initialize the C language using tree_sitter_c
tsc.language()  # ensure the language is loaded
C_LANGUAGE = Language(tsc.language())
c_parser = Parser(C_LANGUAGE)

def get_relevant_cutoff(source_code: str, patch_start_line: int, patch_end_line: int) -> str:
    """
    Given C source code and patch boundaries (1-indexed),
    returns a snippet that includes complete function definitions overlapping
    the patch. If the patch falls inside a function, the snippet is extended to
    include the entire function.
    """
    # Split source code into lines
    lines = source_code.splitlines()
    patch_start_idx = patch_start_line - 1  # convert to 0-indexed
    patch_end_idx = patch_end_line - 1

    # Parse the source code using the custom C parser
    tree = c_parser.parse(source_code.encode('utf8'))
    root = tree.root_node

    # Set initial boundaries to the given patch
    new_start = patch_start_idx
    new_end = patch_end_idx

    def traverse(node):
        yield node
        for child in node.children:
            yield from traverse(child)

    # Traverse the tree and extend the boundaries if the patch falls inside a function
    for node in traverse(root):
        if node.type == 'function_definition':
            func_start = node.start_point[0]
            func_end = node.end_point[0]
            # If function overlaps with the patch region, adjust the boundaries
            if func_start <= patch_end_idx and func_end >= patch_start_idx:
                new_start = min(new_start, func_start)
                new_end = max(new_end, func_end)

    # Return the snippet that covers the adjusted boundaries
    snippet = "\n".join(lines[new_start:new_end+1])
    return snippet


def print_code_with_line_numbers(code_str: str):
    """
    Utility function to print the code along with line numbers.
    """
    lines = code_str.splitlines()
    for i, line in enumerate(lines, start=1):
        print(f"{i:3d}: {line}")


In [ ]:
import tree_sitter_language_pack
from collections import deque
parser = tree_sitter_language_pack.get_parser('c')

In [ ]:
with open('code.c', 'rb') as f:
    code=f.read()

In [ ]:
tu = parser.parse(code)

In [ ]:
root = tu.root_node

In [ ]:
root.children

In [ ]:
root.children[7].children

In [ ]:
root.children[7].children[7].children

In [ ]:
q = deque()
q.append(root)

while q:
    node = q.popleft()
    if node.type == "function_definition":
        if node.parent:
            print(node.type, node.parent.type, node.child_count)
        else:
            print(node.type, None, node.child_count)
    for child in node.children:
        q.append(child)

In [ ]:
c_code = """
#include <stdio.h>

void foo() {
    printf("This is foo\\n");
}

void bar() {
    printf("This is bar\\n");
    // The patch might modify this line
    printf("Modified line\\n");
}

#define MIN 10000

int main() {
    foo();
    bar();
    return 0;
}
""".strip()

# Suppose the patch touches line 10 (1-indexed) which is within 'bar'
patch_start = 10
patch_end = 15

In [ ]:


print("=== Original Code with Line Numbers ===")
print_code_with_line_numbers(c_code)

snippet = get_relevant_cutoff(c_code, patch_start, patch_end)

print("\n=== Extracted Snippet with Line Numbers ===")
print_code_with_line_numbers(snippet)

In [ ]:
patterns = {
    "if_else_block": "(if_statement) @if_block",
    "return_statement": "(return_statement) @return_stmt",
    "switch_block": "(switch_statement) @switch_block",
    "memory_allocation": """
      (call_expression 
         function: (identifier) @mem_func
      ) (#match? @mem_func "^(malloc|calloc|realloc|free)$")
    """,
    "standard_io": """
      (call_expression 
         function: (identifier) @io_func
      ) (#match? @io_func "^(printf|sprintf)$")
    """,
    "conversion_functions": """
      (call_expression 
         function: (identifier) @conv_func
      ) (#match? @conv_func "^(atof|atoi|atol|atoll)$")
    """,
    "standard_library_functions": """
      (call_expression 
         function: (identifier) @lib_func
      ) (#match? @lib_func "^(bsearch|qsort|time|date|memcmp|memcpy|memmove)$")
    """,
    "locale_env_functions": """
      (call_expression 
         function: (identifier) @locale_func
      ) (#match? @locale_func "^(localeconv|getenv|setlocale|strerror|asctime|ctime|gmtime|localtime)$")
    """,
    "random_number_functions": """
      (call_expression 
         function: (identifier) @rand_func
      ) (#match? @rand_func "^(rand|srand)$")
    """,
    "function_parameters": "(parameter_list) @params",
    "union_usage": "(union_specifier) @union_usage",
    "undef_directive": "(preproc_call) @undef_directive (#match? @undef_directive \"^#undef\")",
    "preprocessor_conditionals": "(preproc_call) @preproc_cond (#match? @preproc_cond \"^#(if|ifdef|ifndef|elif|else|endif)\")",
    "goto_statement": "(goto_statement) @goto_stmt",
    "break_statement": "(break_statement) @break_stmt",
    "data_types": """
      ((type_identifier) @type_id (#match? @type_id "^(int|float|short|unsigned|double|char|long)$"))
    """,
    "pointer_usage": "(pointer_declarator) @pointer_usage",
}


In [ ]:
def extract_from_c_code(code: str) -> dict:
    # Initialize results dictionary using the keys in patterns.
    results = {key: [] for key in patterns}
    # Process inline and recursive functions separately.
    results["inline_function"] = []
    results["recursive_function"] = []

    code_bytes = code.encode("utf8")
    parser = Parser(C_LANGUAGE)
    tree = parser.parse(code_bytes)
    root_node = tree.root_node

    # Process each pattern.
    for classification, query_str in patterns.items():
        query = C_LANGUAGE.query(query_str)
        # Group the captures by their capture names.
        for node_list in query.captures(root_node).values():
            for node in node_list:
                # node.start_point and node.end_point are [row, column]
                extracted_text = code_bytes[node.start_byte:node.end_byte].decode("utf8").strip()
                results[classification].append({
                    "classification": classification,
                    "extracted_line": extracted_text,
                    "start_line": node.start_point[0] + 1,
                    "end_line": node.end_point[0] + 1
                })

    query_inline = C_LANGUAGE.query("(function_definition) @func_def")
    for node_list in query_inline.captures(root_node).values():
        for node in node_list:
            func_text = code_bytes[node.start_byte:node.end_byte].decode("utf8")
            # Use regex to check if the function starts with 'inline'
            if re.match(r'^\s*inline\b', func_text):
                results["inline_function"].append({
                    "classification": "inline_function",
                    "extracted_line": func_text.strip(),
                    "start_line": node.start_point[0] + 1,
                    "end_line": node.end_point[0] + 1
                })

    # =============================================================================
    # Process recursive functions:
    # =============================================================================
    query_func_def = C_LANGUAGE.query("(function_definition) @func_def")
    def find_identifier(n):
        # Recursively search for a node of type "identifier" within n.
        if n.type == "identifier":
            return code_bytes[n.start_byte:n.end_byte].decode("utf8")
        for child in n.children:
            res = find_identifier(child)
            if res:
                return res
        return None

    # captures_func_def = query_func_def.captures(root_node)
    for node_list in query_func_def.captures(root_node).values():
        for node in node_list:
            func_name = find_identifier(node)
            if not func_name:
                continue
            # Locate the function's compound statement.
            body_text = ""
            for child in node.children:
                if child.type == "compound_statement":
                    body_text = code_bytes[child.start_byte:child.end_byte].decode("utf8")
                    break
            # Check if the function calls itself.
            if body_text and re.search(r'\b' + re.escape(func_name) + r'\s*\(', body_text):
                func_text = code_bytes[node.start_byte:node.end_byte].decode("utf8")
                results["recursive_function"].append({
                    "classification": "recursive_function",
                    "extracted_line": func_text.strip(),
                    "start_line": node.start_point[0] + 1,
                    "end_line": node.end_point[0] + 1
                })

    return results

In [ ]:
sample_code = r"""
#include <stdio.h>
#include <stdlib.h>

#define MAX 100

inline int add(int a, int b) {
    return a + b;
}

int32_t factorial(int n) {
    if(n <= 1)
        return 1;
    else
        return n * factorial(n - 1);
}

void *my_malloc(size_t size) {
    return malloc(size);
}

int main() {
    int x = 10;
    if(x > 0) {
        printf("Positive\n");
    } else {
        printf("Non-positive\n");
    }
    switch(x) {
        case 10:
            break;
        default:
            break;
    }
    goto end;
    end:
    return 0;
}
"""

extracted = extract_from_c_code(sample_code)
# For pretty printing the results:
import json
print(json.dumps(extracted, indent=2))

In [ ]:
def find_identifier(node, code_bytes):
    """
    Recursively search for an "identifier" node inside the given node.
    Returns the decoded identifier text or None if not found.
    """
    if node.type == "identifier":
        return code_bytes[node.start_byte:node.end_byte].decode("utf8")
    for child in node.children:
        identifier = find_identifier(child, code_bytes)
        if identifier:
            return identifier
    return None

def get_function_boundaries(code: str) -> list:
    """
    Extracts all function boundaries from the provided C code.
    
    Returns a list of dictionaries where each dictionary contains:
      - function_name: Name of the function.
      - start_line: The starting line (1-indexed) of the function definition.
      - end_line: The ending line (1-indexed) of the function definition.
    """
    code_bytes = code.encode("utf8")
    parser = Parser(C_LANGUAGE)
    tree = parser.parse(code_bytes)
    root_node = tree.root_node

    # Query for function_definition nodes.
    query = C_LANGUAGE.query("(function_definition) @func_def")
    boundaries = []
    for node_list in query.captures(root_node).values():
        for node in node_list:
            func_name = find_identifier(node, code_bytes)
            if not func_name:
                continue
            start_line = node.start_point[0] + 1  # tree-sitter rows are 0-indexed
            end_line = node.end_point[0] + 1
            boundaries.append({
                "function_name": func_name,
                "start_line": start_line,
                "end_line": end_line
            })
    return boundaries

def get_c_code_boundaries(code: str) -> dict:
    """
    Returns the boundaries of the entire C code as a dictionary with:
      - start_line: The first line (1-indexed).
      - end_line: The last line (1-indexed).
      
    This is determined by the root node's start and end positions.
    """
    code_bytes = code.encode("utf8")
    parser = Parser(C_LANGUAGE)
    tree = parser.parse(code_bytes)
    root_node = tree.root_node
    return {
        "start_line": root_node.start_point[0] + 1,
        "end_line": root_node.end_point[0] + 1
    }


In [ ]:
if __name__ == "__main__":
    sample_code = r"""
    I am writing this c code
    return x * x;
    to find the boundary. lets
    find it lol

    #include <stdio.h>
    #include <stdlib.h>

    inline int square(int x) {
        return x * x;
    }

    int factorial(int n) {
        if (n <= 1)
            return 1;
        else
            return n * factorial(n - 1);
    }

    int main() {
        int x = 10;
        printf("Hello, World!\n");
        return 0;
    }

    this should not be counted lol
    """
    
    print("Function Boundaries:")
    functions = get_function_boundaries(sample_code)
    for func in functions:
        print(func)
    
    print("\nC Code Boundaries:")
    code_bounds = get_c_code_boundaries(sample_code)
    print(code_bounds)

In [ ]:
import copy

In [ ]:
def print_all_braces_rec(n, left_count, right_count, output, result):

  if left_count >= n and right_count >= n:
    result.append(copy.copy(''.join(output)))
    
  if left_count < n:
    output += '{'
    print_all_braces_rec(n, left_count + 1, right_count, output, result)
    output.pop()

  if right_count < left_count:
    output += '}'
    print_all_braces_rec(n, left_count, right_count + 1, output, result)
    output.pop()

def print_all_braces(n):
  output = []
  result = []
  print_all_braces_rec(n, 0, 0, output, result)
  return result

result = print_all_braces(10)

for rs in result: 
  print(rs)

In [ ]:
result

In [ ]:
arr = [1,2,3,4]

In [ ]:
sorted('word')

In [ ]:
2 << 1

In [ ]:
5 >> 1

In [ ]:
3^5

In [ ]:
for i in range(5):
    print(i)

In [ ]:
import matplotlib.pyplot as plt
import random

# Refilter entries (already removed 2018 in previous step)
entries = [
    # 2021 – Remote Sensing
    {"year": 2021, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "PSSC-RANSAC"},
    {"year": 2021, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "SARPointNet"},
    {"year": 2021, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "RotNET"},
    # 2021 – Medical
    {"year": 2021, "domain": "Medical", "category": "Deformable IR", "method": "HyperMorph"},
    {"year": 2021, "domain": "Medical", "category": "Deformable IR", "method": "Prob. CycleGAN + U-Net"},
    {"year": 2021, "domain": "Medical", "category": "Deformable IR", "method": "CycleMorph"},
    {"year": 2021, "domain": "Medical", "category": "Transformer-based IR", "method": "ViT-V-Net"},
    # 2022 – Remote Sensing
    {"year": 2022, "domain": "Remote Sensing", "category": "Coarse-Fine IR + Fusion", "method": "RFNet"},
    {"year": 2022, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "SAR-SuperPoint"},
    {"year": 2022, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "VLSG-SANet"},
    # 2022 – Medical
    {"year": 2022, "domain": "Medical", "category": "Transformer-based IR", "method": "X-Morpher"},
    {"year": 2022, "domain": "Medical", "category": "Hierarchical / Multi-scale IR", "method": "HCN + BARM"},
    {"year": 2022, "domain": "Medical", "category": "Deformable IR", "method": "TransMorph"},
    {"year": 2022, "domain": "Medical", "category": "Deformable IR", "method": "MambaMorph"},
    {"year": 2022, "domain": "Medical", "category": "Deformable IR", "method": "Swin-VoxelMorph"},
    {"year": 2022, "domain": "Medical", "category": "Multimodal Deformable IR", "method": "Resblock + CascadeNet + STF"},
    {"year": 2022, "domain": "Medical", "category": "Implicit IR", "method": "MLP + SIREN"},
    # 2023 – Remote Sensing
    {"year": 2023, "domain": "Remote Sensing", "category": "Coarse-Fine IR + Fusion", "method": "MURF"},
    {"year": 2023, "domain": "Remote Sensing", "category": "Coarse-Fine IR", "method": "Deep Ordinal Regression"},
    {"year": 2023, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "Deep corr. + Attn"},
    {"year": 2023, "domain": "Remote Sensing", "category": "Feature-based IR", "method": "FMPR-Net"},
    # 2023 – Medical
    {"year": 2023, "domain": "Medical", "category": "Deformable IR", "method": "VoxelMorph (fast 3D)"},
    {"year": 2023, "domain": "Medical", "category": "Spatio-temporal Deformable IR", "method": "Seq2Morph"},
    {"year": 2023, "domain": "Medical", "category": "Foundation-model IR", "method": "UniGradICON"},
    {"year": 2023, "domain": "Medical", "category": "Deformable IR", "method": "ModeT"},
    {"year": 2023, "domain": "Medical", "category": "Affine IR", "method": "PVIT-AIR"},
    # 2024 – Remote Sensing
    {"year": 2024, "domain": "Remote Sensing", "category": "Multimodal Comprehension", "method": "EarthGPT"},
    {"year": 2024, "domain": "Remote Sensing", "category": "Vision-Language Align.", "method": "LHRS-Bot"},
    {"year": 2024, "domain": "Remote Sensing", "category": "Visual-Prompted IR", "method": "EarthMarker"},
    {"year": 2024, "domain": "Remote Sensing", "category": "IR Agents (Tool-aug.)", "method": "GeoLLM-Engine"},
    # 2024 – Medical
    {"year": 2024, "domain": "Medical", "category": "Deformable + Multitemporal IR", "method": "D-LinkNet + DeepResUNet"},
    {"year": 2024, "domain": "Medical", "category": "Deformable IR", "method": "H-ViT"},
    {"year": 2024, "domain": "Medical", "category": "Deformable IR", "method": "LLM-Morph"},
    {"year": 2024, "domain": "Medical", "category": "Deformable IR", "method": "LLaMA-Reg"},
    {"year": 2024, "domain": "Medical", "category": "Deformable IR", "method": "SAMReg"},
    {"year": 2024, "domain": "Medical", "category": "Rigid + Deformable IR", "method": "LapIRN"},
]

# Setup categories and y positions (compressed)
categories = sorted({e["category"] for e in entries})
y_pos = {cat: i for i, cat in enumerate(categories)}

fig, ax = plt.subplots(figsize=(12, 5))

# Plot nodes
for e in entries:
    x = e["year"] + (-0.2 if e["domain"] == "Remote Sensing" else 0.2)
    y = y_pos[e["category"]]
    marker = 'o' if e["domain"] == "Remote Sensing" else '^'
    ax.scatter(x, y, marker=marker, s=100, zorder=3)

# Draw edges within each category
for cat in categories:
    cat_entries = [e for e in entries if e["category"] == cat]
    cat_entries.sort(key=lambda e: e["year"])
    xs = [e["year"] + (-0.2 if e["domain"] == "Remote Sensing" else 0.2) for e in cat_entries]
    ys = [y_pos[cat]] * len(cat_entries)
    if len(xs) > 1:
        ax.plot(xs, ys, linewidth=1.0, zorder=2)

# Add text labels with staggered offsets to avoid overlap
for cat in categories:
    cat_entries = [e for e in entries if e["category"] == cat]
    cat_entries.sort(key=lambda e: (e["year"], e["domain"]))
    n = len(cat_entries)
    for j, e in enumerate(cat_entries):
        x = e["year"] + (-0.2 if e["domain"] == "Remote Sensing" else 0.2)
        base_y = y_pos[cat]
        offset = (j - (n - 1) / 2) * 0.2 # staggered vertical offset
        ax.text(
            x, base_y + 0.15 + offset,
            e["method"],
            rotation=45, ha='right', va='bottom', fontsize=8
        )

# Formatting
ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(categories, fontsize=9)
ax.set_xlabel("Year")
ax.set_xlim(2020.7, 2024.5)

# Remove gridlines and axes spines
ax.grid(False)
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)
ax.tick_params(left=False, bottom=True)

plt.tight_layout()
# plt.savefig('ir_evolution_cleaned_v2.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import io, numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- Data -------------------------------------------------
raw_table = """
Type\tCitation\tParam\tCategory
General\t2024\t14092815\tfeature
Medical (locked)\t2024\t34523650\tDeformable
Medical\t2024\t8897107\tDeformable through Spatial Landmark Detection
Medical\t2024\t8922444\tdeformable, affine
General\t2023\t303424\tfeature
Medical\t2024\t38449920\tDeformable Non-Rigid Registration Volumetric; Coarse-to-Fine
Medical/Histological\t2024\t450564\tdeformable
Medical (locked)\t2024\t1491\tdeformable
General/Medical\t2024\t1076964\tdeformable
Medical\t2024\t7310000\tMulti-modal deformable registration
Medical\t2024\t53007765\tdeformable
Medical\t2024\t330000\tDeformable IR
Medical\t2024\t896551\tDeformable IR
Medical\t2024\t328791\tAffine IR
Medical\t2024\t615076\tDeformable and Rigid IR
Medical\t2024\t9640000\tMulti-modal deformable registration
Medical\t2024\t163737623\tDeformable IR
Remote Sensing\t2024\t7503968\tCoarse-Fine IR
Medical\t2024\t133379\tdeformable IR
Remote Sensing\t2024\t7216893\tCoarse-Fine IR
Remote Sensing\t2024\t31096129\tdeformable
Remote Sensing\t2024\t4902300\tFeature based IR, Keypoint Matching
General/Medical\t2023\t960998\tCoarse-Fine IR
General/Medical\t2023\t165659\trigid and deformable
Medical\t2023\t112350291\tdeformable IR
General\t2023\t8630497\tDeformable/affine IR
General\t2023\t67319427\tfeature
Medical\t2023\t133379\tdeformable IR
Medical\t2023\t18290716\tdeformable IR
Medical\t2023\t233476\tdiffeomorphic deformable IR
Remote Sensing\t2023\t7549640\tfeature
Medical\t2024\t10180163\tDeformable IR
Medical\t2023\t3222\tAffine IR
Medical\t2023\t133379\tDeformable IR
Medical\t2023\t16318691\tDeformable IR
Medical\t2023\t133379\tDeformable IR
Medical\t2023\t53010687\tdeformable
Medical\t2023\t13140670\tdeformable
Medical\t2023\t5706508\tCoarse-Fine IR
General\t2023\t77430931\tdeformable IR
Medical\t2023\t1029670\tDeformable IR
Medical\t2023\t526448\tDeformable IR, multimodal volumetric
Medical\t2023\t4085458\tfeature
Remote Sensing\t2023\t7151276\tdeformable
General\t2022\t1962429\tcross modal deformable IR
Medical/General/Remote Sensing\t2023\t960998\tdeformable affine IR
Medical\t2022\t12030208\tAffine Registration
Medical\t2022\t133379\tDeformable IR
Medical\t2022\t7976611\tDeformable Registration
Medical/General\t2022\t90672387\tDeformable Registration
Medical\t2022\t11721891\tDeformable
Medical\t2022\t21002443\tDeformable
Medical\t2022\t16049667\tDeformable
Medical\t2022\t327331\tDeformable
Medical\t2022\t1029670\tDeformable
Medical\t2022\t45481080\tDeformable
Medical\t2022\t18290716\tDeformable
Remote Sensing\t2022\t13841096\tFeature-based
Medical\t2022\t5827427\tDeformable
Medical\t2022\t336080\tdiffeomorphic
Medical\t2022\t1248777\tDeformable, Feature-based
Medical\t2022\t608112\tDeformable
Medical\t2022\t108611336\tDeformable
Remote Sensing\t2022\t13600229\tDeformable
Medical\t2021\t2374072\tDeformable, Groupwise
Medical\t2021\t45510153\tDeformable
Medical\t2021\t722598\tDeformable Image Registration
Medical\t2021\t31560079\tDeformable Image Registration
Medical\t2021\t608112\tDeformable Image Registration
Medical\t2021\t23007954\tDeformable
Remote Sensing\t2021\t40847185\tfeature
Medical\t2021\t491668\tDeformable
Medical\t2021\t1739270\tDeformable
Medical\t2021\t11774539\tDeformable
Remote Sensing\t2021\t95266\tDeformable
"""
df = pd.read_csv(io.StringIO(raw_table), sep="\t")

# --- classification functions ----------
def get_type_group(t: str) -> str:
    t_low = t.lower()
    if "remote sensing" in t_low:
        return "Remote Sensing"
    elif "medical" in t_low:
        return "Medical"
    else:
        return "Other"

def get_broad_cat(c: str) -> str:
    c_low = c.lower()
    if "feature" in c_low:
        return "Feature‑based"
    elif "deformable" in c_low:
        return "Deformable"
    else:
        return "Other"

df["TypeGroup"] = df["Type"].apply(get_type_group)
df["BroadCat"] = df["Category"].apply(get_broad_cat)
df["Param"] = pd.to_numeric(df["Param"], errors="coerce")
df.dropna(subset=["Param"], inplace=True)

# keep only two domains
df = df[df["TypeGroup"].isin(["Medical", "Remote Sensing"])]
df["LogParam"] = np.log10(df["Param"])

years = sorted(df["Citation"].unique())  # 2021-2024
cats  = ["Feature‑based", "Deformable", "Other"]
types = ["Medical", "Remote Sensing"]

data_for_boxes = []
positions      = []
pos_map        = {}
pos_counter = 0
group_width = len(cats)*len(types)
gap = 0.5  # squeeze x‑axis

for year in years:
    for cat_idx, cat in enumerate(cats):
        for typ_idx, typ in enumerate(types):
            subset = df[(df["Citation"]==year) & (df["BroadCat"]==cat) & (df["TypeGroup"]==typ)]
            if not subset.empty:
                data_for_boxes.append(subset["LogParam"].values)
                current_pos = pos_counter + cat_idx*len(types)+typ_idx
                positions.append(current_pos)
                pos_map[(year, cat, typ)] = current_pos
    pos_counter += group_width + gap

# --- plotting ----------
plt.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(figsize=(10,6))

box = ax.boxplot(
    data_for_boxes,
    positions=positions,
    widths=0.4,
    patch_artist=True,
    showfliers=False
)

# colorize boxes by BroadCat
cat_colors = {"Feature‑based":"#4e79a7", "Deformable":"#f28e2b", "Other":"#59a14f"}
for b, pos in zip(box["boxes"], positions):
    # need to figure cat of that pos
    # reverse map
    cat_of_box = None
    for (yr, cat, typ), p in pos_map.items():
        if p == pos:
            cat_of_box = cat
            break
    b.set_facecolor(cat_colors.get(cat_of_box,"#bab0ac"))
    b.set_alpha(0.55)
    b.set_edgecolor('black')

# whiskers color
for whisk in box['whiskers']:
    whisk.set_color('black')
for cap in box['caps']:
    cap.set_color('black')
for median in box['medians']:
    median.set_color('black')

# scatter overlay
for _, row in df.iterrows():
    combo = (row["Citation"], row["BroadCat"], row["TypeGroup"])
    if combo in pos_map:
        jitter = (np.random.rand()-0.5)*0.3
        marker = 'o' if row["TypeGroup"]=="Medical" else '^'
        ax.scatter(pos_map[combo]+jitter, row["LogParam"],
                   marker=marker, s=40, edgecolors='k', linewidths=0.4, alpha=0.7)

# x ticks
centre_points = []
labels   = []
pos_counter = 0
for y in years:
    centre_points.append(pos_counter + (group_width-1)/2)
    labels.append(str(y))
    pos_counter += group_width + gap
ax.set_xticks(centre_points)
ax.set_xticklabels(labels)

ax.set_ylabel("log\u2081\u2080(Parameter count)")
# ax.set_title("Image‑registration models: parameter distribution by year/domain", fontsize=18)

# legends
legend_elems = [
    Line2D([0],[0], marker='o', linestyle='', label='Medical', markeredgecolor='k',markerfacecolor='white'),
    Line2D([0],[0], marker='^', linestyle='', label='Remote Sensing', markeredgecolor='k',markerfacecolor='white'),
    Line2D([0],[0], color=cat_colors["Feature‑based"], lw=10, label='Feature‑based IR'),
    Line2D([0],[0], color=cat_colors["Deformable"], lw=10, label='Deformable IR'),
    Line2D([0],[0], color=cat_colors["Other"], lw=10, label='Other IR')
]
ax.legend(handles=legend_elems, loc='upper left', bbox_to_anchor=(1,1))
fig.tight_layout()
